# YOLO26n Training on Colab (T4)

Trains **YOLO26n** as a third comparison model alongside the Phase 3a baselines
(Faster R-CNN, YOLOv8n), on the same SOURCE split (India + Japan), same seed (42),
same dataset config -- only the architecture and batch size differ.

**Why the smaller batch (8, vs. YOLOv8n's 16):** YOLO26 is a newer, less-tuned
architecture on this pipeline, so a smaller batch reduces memory pressure and
per-step latency variance on a free T4. Combined with `--chunk-epochs`, each cell
finishes faster and nothing semi-trained is ever lost -- rerunning the same command
resumes exactly where it left off (true resume: optimizer/EMA/LR-scheduler state
preserved, not a naive restart).

**This notebook gets its CODE from a small zip you upload directly in Step 3b**,
not from whatever is bundled in your Drive's `rdd_bundle.zip` -- this sidesteps the
stale-code problem entirely (an old cached bundle silently missing new flags).

## Step 1 — Confirm a GPU is actually attached

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "No CUDA GPU. Set Runtime -> Change runtime type -> GPU, then rerun."
)
print("device:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Step 2 — Mount Google Drive

Needed for the **dataset** (large, unchanged from earlier sessions) and for saving
checkpoints between chunks. When the popup appears: **Connect to Google Drive** →
choose your account → tick **all** permission boxes → **Continue**.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 3a — Extract the DATA bundle (code inside it will be ignored/overwritten in Step 3b)

Edit `BUNDLE_ZIP` if your zip is not at `My Drive/rdd_bundle.zip`. This always does a
clean extract (no skip-if-exists), so a half-old directory can never linger.

In [ ]:
import zipfile, os, shutil, time
from pathlib import Path

BUNDLE_ZIP = '/content/drive/MyDrive/rdd_bundle.zip'
REPO = Path('/content/road-damage-detection')

assert Path(BUNDLE_ZIP).exists(), f"Not found: {BUNDLE_ZIP} -- check the path in your Drive"

if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True, exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(REPO)
print(f"Extracted data bundle in {time.time() - t0:.0f}s")

os.chdir(REPO)
print("cwd:", os.getcwd())

## Step 3b — Upload the CODE patch and overwrite src/ + config/

Run this cell, then in the file picker choose **`dist/code_patch.zip`** from your
local repo (rebuild it any time your local code changes with the snippet in
`README.md` / ask your assistant to regenerate it). This guarantees the code running
below is exactly what's on your machine right now, regardless of what the Drive data
bundle's own (possibly stale) copy of `src/`/`config/` contains.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()   # select dist/code_patch.zip
patch_name = next(iter(uploaded))

with zipfile.ZipFile(patch_name) as zf:
    zf.extractall('/content/road-damage-detection')

print("patched:", patch_name)
print()
!grep -n "base-model" src/models/train_yolo.py

The last line above must print the `--base-model` argument definition. If it prints nothing, the upload picked the wrong file -- re-run this cell.

## Step 4 — Install dependencies

In [ ]:
!pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 5 — Point the dataset configs at Colab paths

The YAMLs ship with Windows paths (restored verbatim by every extract) -- ultralytics
resolves relative dataset paths against its own settings dir, not the repo, so this
must be regenerated **after** Step 3a, every session.

In [ ]:
!python src/data/write_dataset_configs.py --data-root /content/road-damage-detection/data/processed
print()
!cat config/dataset_source.yaml

## Step 6 — Verify the data arrived intact

Counts must match `data/split_report.json` (12,748 train / 2,732 val / 2,732 test).

In [ ]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    imgs = len(list(Path(f'data/processed/source/images/{split}').glob('*.jpg')))
    lbls = len(list(Path(f'data/processed/source/labels/{split}').glob('*.txt')))
    print(f"{split:5s}: {imgs:6d} images  {lbls:6d} labels")

expected = {'train': 12748, 'val': 2732, 'test': 2732}
for split, n in expected.items():
    got = len(list(Path(f'data/processed/source/images/{split}').glob('*.jpg')))
    assert got == n, f"{split}: expected {n} images, found {got} -- upload incomplete"
print("Counts match split_report.json.")

import yaml
cfg = yaml.safe_load(open('config/dataset_source.yaml'))
for key in ('train', 'val', 'test'):
    p = Path(cfg[key])
    assert p.is_absolute() and p.exists(), (
        f"config/dataset_source.yaml {key} points at {cfg[key]} which does not exist."
    )
print("Dataset config paths resolve correctly.")

## Step 7 — Define helpers AND restore previous progress

> **Run this cell in EVERY session, before any training chunk.**
> Defines `save_results()`/`progress()` (kernel memory only -- lost on reconnect),
> and restores checkpoints from Drive so a recycled VM doesn't silently restart
> training from epoch 0.

In [ ]:
import shutil, os
from pathlib import Path

def restore_results():
    src_root = Path('/content/drive/MyDrive/rdd_results')
    if not src_root.exists():
        print("No saved results in Drive yet -- this is a fresh start.")
        return
    for name in ['runs', 'results']:
        src = src_root / name
        if src.exists():
            dst = Path('experiments') / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("restored", dst)

def save_results():
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        out = drive_root / 'rdd_results'
        out.mkdir(parents=True, exist_ok=True)
        for src in [Path('experiments/runs'), Path('experiments/results')]:
            if src.exists():
                dst = out / src.name
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
        print("saved to Drive:", out)
        return out
    print("Drive not mounted -- packing results for download instead.")
    archive = shutil.make_archive('/content/rdd_results', 'zip', 'experiments')
    print("archive:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
    from google.colab import files
    files.download(archive)
    return Path(archive)

def progress():
    import csv as _csv, yaml as _yaml
    for run in sorted(Path('experiments/runs').glob('*')):
        rc, ay = run / 'results.csv', run / 'args.yaml'
        if rc.exists():
            done = max(0, sum(1 for _ in _csv.reader(open(rc))) - 1)
            total = (_yaml.safe_load(open(ay)) or {}).get('epochs', '?') if ay.exists() else '?'
            print(f"  {run.name}: {done}/{total} epochs")

restore_results()
print("\nhelpers ready. current progress:")
progress()

## Step 8 — Smoke test (recommended, ~1 min on T4)

Logged with phase `3a-smoke` -- a pipeline artifact, never a reportable result.

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source_subset.yaml \
    --name yolo26n_smoke \
    --base-model yolo26n.pt \
    --batch 8 --epochs 2 --smoke --device 0

## Step 9 — Train YOLO26n: 100 epochs as 10 chunks of 10

Batch 8 (half of YOLOv8n's 16), 10-epoch chunks (half of YOLOv8n's 20) -- shorter,
safer cells per your request. **Re-run the SAME cell below repeatedly** until it
reports training complete; each rerun resumes exactly where the last one stopped.

> **Returning to a new session?** Re-run Steps 1–7 first (GPU, mount, extract data,
> upload code patch, install, configs, verify, helpers+restore). Jumping straight to
> this cell in a fresh kernel fails with `NameError: save_results is not defined`,
> and if the VM was recycled, would restart training from epoch 0.

In [ ]:
!python src/models/train_yolo.py \
    --data config/dataset_source.yaml \
    --name yolo26n_source \
    --base-model yolo26n.pt \
    --batch 8 \
    --chunk-epochs 10 \
    --device 0

save_results()   # persist this chunk to Drive before anything can drop

### Progress check (run any time)

In [ ]:
progress()

## Step 10 — Review the result

Once Step 9 reports `Training already complete`, a real (non-smoke) row is already
logged in `experiment_log.csv` with `model: yolo26n`.

In [ ]:
import pandas as pd

log = pd.read_csv('experiments/results/experiment_log.csv')
real = log[~log['phase'].astype(str).str.contains('smoke')]
real = real[real['model'] == 'yolo26n']

cols = ['run_id', 'model', 'dataset', 'split', 'num_images', 'map50', 'map50_95',
        'ap_D00', 'ap_D10', 'ap_D20', 'ap_D40', 'precision', 'recall', 'f1',
        'latency_ms', 'fps', 'param_count']
print(real[cols].to_string(index=False) if not real.empty else "No non-smoke yolo26n row yet.")

## Final step — Save everything back to Drive

Then download `experiments/results/experiment_log.csv` and drop it in your local
repo's `incoming/` folder -- tell your assistant once it's there so the config hash
can be verified before merging.

In [ ]:
out = save_results()

if out.is_dir():
    print("\nContents:")
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(" ", p.relative_to(out), f"{p.stat().st_size / 1e6:.1f} MB")